# 16장. 실험 기록과 최종 평가

실험 하나를 다시 만드는 데 필요한 것을 기록하고, 같은 기록으로 다시 실행했을 때 결과가 얼마나 달라지는지 잰다. 검증 자료로 고른 모델을 시험용 파일로 한 번 평가하고, 시간이 지나 자료가 달라질 때 모델을 다시 학습시키는 기준을 정한다.

**준비:** 9장의 학습용 파일 두 개와 시험용 파일 `t10k-images-idx3-ubyte.gz`, `t10k-labels-idx1-ubyte.gz`(https://github.com/zalandoresearch/fashion-mnist), 11장의 `ratings_train.txt`와 시험용 파일 `ratings_test.txt`(https://github.com/e9t/nsmc), 8장의 아파트 실거래가 파일 두 개(2023년, 2024년 계약분)를 이 노트북과 같은 폴더에 둔다. 시험용 파일은 16.4절의 셀 두 개에서만 읽는다. CPU에서 실행하고 전체 실행에 15분쯤 걸린다.

## 16.1 실험을 기록하기: 시드, 환경, 설정, 체크포인트

In [ ]:
import copy, glob, gzip, hashlib, json, math, os, platform, re, sys, time
from collections import Counter
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn

torch.set_num_threads(2)

def read_idx(path, offset):
    """idx 형식의 파일을 읽는다. 앞의 offset바이트는 머리말이다."""
    with gzip.open(path) as f:
        return np.frombuffer(f.read(), np.uint8, offset=offset)

def fingerprint(path):
    """파일 내용의 SHA-256 해시에서 앞 12글자. 한 바이트만 달라도 값이 달라진다."""
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()[:12]

FILES = ["train-images-idx3-ubyte.gz", "train-labels-idx1-ubyte.gz"]
images = read_idx(FILES[0], 16).reshape(-1, 28, 28)
labels = read_idx(FILES[1], 8).astype(np.int64)
order = np.random.default_rng(0).permutation(60000)           # 9장과 같은 나누기
train_rows, valid_rows = order[:6000], order[50000:]

def tensors(pixels, marks, mean=None, std=None):
    """이미지를 (장수, 1, 28, 28) 모양의 실수 텐서로 바꾸고 표준화한다."""
    x = torch.tensor(pixels / 255, dtype=torch.float32)[:, None]
    if mean is None:
        mean, std = x.mean().item(), x.std().item()
    return (x - mean) / std, torch.tensor(marks), mean, std

X_tr, y_tr, pixel_mean, pixel_std = tensors(images[train_rows], labels[train_rows])
X_va, y_va, _, _ = tensors(images[valid_rows], labels[valid_rows],
                           pixel_mean, pixel_std)
print(X_tr.shape, X_va.shape)
for name in FILES:
    print(name, fingerprint(name))

In [ ]:
def make_cnn():
    """9.3절의 모델: 합성곱(16) - 풀링 - 합성곱(32) - 풀링 - 선형."""
    return nn.Sequential(
        nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(), nn.Linear(32 * 7 * 7, 10))

def scores_of(net, x):
    """소프트맥스에 넣기 전의 점수. 1,000장씩 나눠 계산한다."""
    kind = next(net.parameters()).dtype
    net.eval()
    with torch.no_grad():
        parts = [net(x[k:k + 1000].to(kind)) for k in range(0, len(x), 1000)]
    return torch.cat(parts).float()

def score(net):
    """검증 이미지 10,000장의 정확도(%)."""
    return (scores_of(net, X_va).argmax(dim=1) == y_va).float().mean().item() * 100

CONFIG = dict(model="합성곱 16-32", seed=0, epochs=10, batch=64, lr=0.001,
              optimizer="Adam", threads=2, dtype="float32", device="cpu", stop=None)

def learn(config, start, keep):
    kind, device = getattr(torch, config["dtype"]), config["device"]
    torch.manual_seed(config["seed"])                       # 가중치의 초깃값을 정한다
    net = make_cnn().to(device, kind)
    x, y = X_tr.to(device, kind), y_tr.to(device)
    optimizer = torch.optim.Adam(net.parameters(), lr=config["lr"])
    shuffler = torch.Generator().manual_seed(config["seed"])   # 섞는 순서를 정한다
    first, step = 0, 0
    if start is not None:                                   # 체크포인트에서 이어 간다
        net.load_state_dict(start["model"])
        if "optimizer" in start:
            optimizer.load_state_dict(start["optimizer"])
        if "shuffler" in start:
            shuffler.set_state(start["shuffler"])
        first = start["epoch"]
    for epoch in range(first, config["epochs"]):
        net.train()
        mixed = torch.randperm(len(x), generator=shuffler)
        for begin in range(0, len(x), config["batch"]):
            rows = mixed[begin:begin + config["batch"]]
            optimizer.zero_grad()
            F.cross_entropy(net(x[rows]), y[rows]).backward()
            optimizer.step()
            step += 1
            if step == config["stop"]:                      # 정한 걸음에서 멈춘다
                return net.cpu()
        if keep is not None:
            keep(epoch + 1, net, optimizer, shuffler)
    return net.cpu()

def run(config, start=None, keep=None):
    """config대로 학습시킨다. start: 이어 갈 체크포인트, keep: 에폭마다 부를 함수."""
    torch.set_num_threads(config["threads"])
    net = learn(config, start, keep)
    torch.set_num_threads(2)
    return net

def environment():
    """결과에 영향을 줄 수 있는 환경."""
    return dict(python=sys.version.split()[0], numpy=np.__version__,
                torch=torch.__version__, machine=platform.machine(),
                cuda=torch.cuda.is_available())

lap = time.perf_counter()
base = run(CONFIG)
took = {"10에폭 학습 한 번": time.perf_counter() - lap}
record = dict(config=CONFIG, environment=environment(),
              files={name: fingerprint(name) for name in FILES},
              split="default_rng(0).permutation(60000), 앞 6000 학습, 뒤 10000 검증",
              result={"검증 정확도": round(score(base), 2)})
with open("run_seed0.json", "w", encoding="utf-8") as f:
    json.dump(record, f, ensure_ascii=False, indent=1)
print(open("run_seed0.json", encoding="utf-8").read())

In [ ]:
def checkpoint(epoch, net, optimizer, shuffler):
    """이어서 학습하는 데 필요한 것: 가중치, 옵티마이저와 섞는 순서의 상태, 에폭."""
    return copy.deepcopy(dict(model=net.state_dict(), optimizer=optimizer.state_dict(),
                              shuffler=shuffler.get_state(), epoch=epoch))

def keep_fifth(epoch, net, optimizer, shuffler):
    if epoch == 5:
        torch.save(checkpoint(epoch, net, optimizer, shuffler), "ckpt_seed0_e05.pt")

whole = run(CONFIG, keep=keep_fifth)                   # 10에폭을 끊지 않고 학습시킨다
saved = torch.load("ckpt_seed0_e05.pt")
print(list(saved), saved["epoch"])
print(len(saved["model"]), sum(t.numel() for t in saved["model"].values()))
slot = saved["optimizer"]["state"][0]                 # 첫 번째 파라미터에 딸린 상태
print(list(slot), int(slot["step"]), tuple(slot["exp_avg"].shape))
print(tuple(saved["shuffler"].shape),
      round(os.path.getsize("ckpt_seed0_e05.pt") / 1024), "KB")

In [ ]:
def pad(text, width):
    wide = sum(ord(ch) > 0x1100 for ch in text)
    return text + " " * max(width - len(text) - wide, 1)

def compare(name, net, ref=None):
    """ref(기본은 끊지 않고 학습시킨 모델)와 비교한다.
    (가중치의 가장 큰 차이, 검증 정확도, 예측이 달라진 이미지의 수)"""
    ref = whole if ref is None else ref
    gap = max((p.double() - q.double()).abs().max().item()
              for p, q in zip(net.parameters(), ref.parameters()))
    mine = scores_of(net, X_va).argmax(dim=1)
    moved = int((mine != scores_of(ref, X_va).argmax(dim=1)).sum())
    mark = (mine == y_va).float().mean().item() * 100
    print(pad(name, 28) + f"{gap:>10.2e} {mark:>10.2f} {moved:>10,}")
    return gap, mark, moved

WAYS = {"모두 넣음": ["model", "optimizer", "shuffler"],
        "가중치만 넣음": ["model"],
        "섞는 순서의 상태를 뺌": ["model", "optimizer"],
        "옵티마이저의 상태를 뺌": ["model", "shuffler"]}
HEAD = "가중치 차이  검증 정확도  달라진 예측"
resumed = {}
print(pad("5에폭에서 이어서 학습", 28) + HEAD)
print(pad("끊지 않고 학습(기준)", 28) + f"{0:>10.2e} {score(whole):>10.2f} {0:>10,}")
for name, parts in WAYS.items():
    start = {key: saved[key] for key in parts + ["epoch"]}
    resumed[name] = compare(name, run(CONFIG, start=start))

## 16.2 같은 기록으로 다시 실행했을 때 결과가 달라지는 정도

In [ ]:
VARIANTS = {"같은 기록으로 다시": {}, "스레드 1개": {"threads": 1},
            "64비트 실수": {"dtype": "float64"}}
changed = {}
print(pad("환경을 바꿔 다시 실행", 28) + HEAD)
for name, change in VARIANTS.items():
    changed[name] = compare(name, run({**CONFIG, **change}), base)

In [ ]:
STOPS = [1, 2, 5, 10, 20, 50, 94, 188, 470, 940]      # 94걸음이 1에폭, 940걸음이 10에폭

def flat(net):
    return torch.cat([p.detach().double().flatten() for p in net.parameters()])

lap = time.perf_counter()
apart = {"스레드 1개": [], "64비트 실수": []}
for stop in STOPS:
    ref = flat(run({**CONFIG, "stop": stop}))
    for name in apart:
        other = flat(run({**CONFIG, **VARIANTS[name], "stop": stop}))
        apart[name].append((ref - other).abs().max().item())
took["걸음마다 비교"] = time.perf_counter() - lap
print("걸음      스레드 1개   64비트 실수")
for k, stop in enumerate(STOPS):
    print(f"{stop:>4} {apart['스레드 1개'][k]:>14.2e} {apart['64비트 실수'][k]:>13.2e}")
a, b, c = torch.tensor([1e8, 1.0, -1e8], dtype=torch.float32)     # 32비트 실수 세 개
print(float((a + b) + c), float((a + c) + b))                  # 더하는 순서만 다르다

In [ ]:
MISSING = {"시드를 1로": {"seed": 1}, "배치 크기를 32로": {"batch": 32},
           "에폭을 9로": {"epochs": 9}, "학습률을 0.002로": {"lr": 0.002}}
print(pad("기록에서 한 항목이 다를 때", 28) + HEAD)
for name, change in MISSING.items():
    changed[name] = compare(name, run({**CONFIG, **change}), base)

In [ ]:
if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True     # 합성곱을 정해진 방법으로 계산한다
    torch.backends.cudnn.benchmark = False        # 실행마다 방법을 새로 고르지 않는다
    print(torch.cuda.get_device_name(0))
    on_gpu = run({**CONFIG, "device": "cuda"})
    changed["GPU"] = compare("GPU에서 실행", on_gpu, base)
    twice = run({**CONFIG, "device": "cuda"})
    changed["GPU 두 번"] = compare("GPU에서 한 번 더", twice, on_gpu)
else:
    print("GPU가 없다. 이 셀의 비교는 실행하지 않았다.")

## 16.3 고르는 데 쓴 자료의 숫자와 새 자료의 숫자: 시드와 체크포인트

In [ ]:
SEEDS, EPOCHS = 10, 20
os.makedirs("ckpt", exist_ok=True)
right = np.zeros((SEEDS, EPOCHS, len(y_va)), dtype=bool)     # (시드, 에폭, 이미지)

lap = time.perf_counter()
for seed in range(SEEDS):
    def keep(epoch, net, optimizer, shuffler, seed=seed):
        """에폭마다 검증 이미지를 맞혔는지 적고 가중치를 파일로 저장한다."""
        right[seed, epoch - 1] = (scores_of(net, X_va).argmax(dim=1) == y_va).numpy()
        torch.save(net.state_dict(), f"ckpt/seed{seed}_e{epoch:02d}.pt")
    run({**CONFIG, "seed": seed, "epochs": EPOCHS}, keep=keep)
took["20에폭 학습 10번"] = time.perf_counter() - lap
acc = right.mean(axis=2) * 100                               # 검증 정확도(시드, 에폭)
print("에폭    평균    최소    최대")
for epoch in [1, 5, 10, 15, 20]:
    col = acc[:, epoch - 1]
    print(f"{epoch:>3} {col.mean():>8.2f} {col.min():>7.2f} {col.max():>7.2f}")
print(" ".join(f"{a:.2f}" for a in acc[:, -1]))
print(round(float(acc[0, 9]), 2), len(glob.glob("ckpt/*.pt")),
      round(sum(os.path.getsize(f) for f in glob.glob("ckpt/*.pt")) / 1e6, 1), "MB")

In [ ]:
last = acc[:, -1]                                            # 20에폭의 검증 정확도
print(int(last.argmax()), round(float(last.max()), 2), round(float(np.median(last)), 2),
      round(float(last.mean()), 2), round(float(last.min()), 2))

def chosen(a, rule):
    """정확도 표 a(시드, 에폭)에서 rule대로 고른 칸의 시드 번호와 에폭 번호."""
    if rule == "시드":                              # 20에폭에서 가장 좋은 시드 하나
        return [int(a[:, -1].argmax())], [a.shape[1] - 1]
    if rule == "체크포인트":                        # 시드마다 가장 좋은 에폭
        return np.arange(len(a)), a.argmax(axis=1)
    seed, epoch = np.unravel_index(a.argmax(), a.shape)     # 200칸에서 가장 좋은 하나
    return [int(seed)], [int(epoch)]

def optimism(rule, n_pick, draws=200):
    """검증 이미지를 무작위로 나눠 n_pick장으로 고르고, 고른 것이 20에폭 평균보다
    얼마나 높은지를 고른 쪽과 쓰지 않은 쪽에서 잰다. draws번 반복한 평균."""
    rng, rows = np.random.default_rng(0), []
    for _ in range(draws):
        mixed = rng.permutation(right.shape[2])
        a = right[:, :, mixed[:n_pick]].mean(axis=2) * 100   # 고르는 데 쓴 이미지
        b = right[:, :, mixed[n_pick:]].mean(axis=2) * 100   # 쓰지 않은 이미지
        seeds, epochs = chosen(a, rule)
        rows.append((a[seeds, epochs].mean() - a[:, -1].mean(),
                     b[seeds, epochs].mean() - b[:, -1].mean()))
    return np.mean(rows, axis=0)

RULES = ["시드", "체크포인트", "시드와 체크포인트"]
PICKS = [500, 2000, 5000]
lifted = {}
print(pad("고르는 방법", 20) + "고른 장수  고른 쪽  쓰지 않은 쪽    차이")
for rule in RULES:
    for n_pick in PICKS:
        lifted[rule, n_pick] = optimism(rule, n_pick)
        here, there = lifted[rule, n_pick]
        print(pad(rule, 20) + f"{n_pick:>8,} {here:>+8.2f} {there:>+12.2f} "
              f"{here - there:>+7.2f}")

## 16.4 남겨 둔 자료로 한 번만 평가하기

In [ ]:
def bins_of(p, y, bins=10):
    """확신을 0.1 간격의 구간으로 나눠 (장수, 평균 확신, 맞힌 비율)을 구한다."""
    sure, guess = p.max(dim=1)
    hit = (guess == y).float()
    where = torch.clamp((sure * bins).long(), max=bins - 1)
    return [(b / bins, int((where == b).sum()), sure[where == b].mean().item(),
             hit[where == b].mean().item()) for b in range(bins) if (where == b).any()]

def ece(p, y):
    """기대 보정 오차(%)."""
    return sum(n / len(y) * abs(hit - sure) for _, n, sure, hit in bins_of(p, y)) * 100

def nll(p, y):
    return -torch.log(p[torch.arange(len(y)), y].clamp_min(1e-12)).mean().item()

TEMPS = torch.exp(torch.linspace(math.log(0.3), math.log(5), 400))   # 온도 후보 400개

def fit_T(z, y):
    """점수 z를 T로 나눈 뒤의 손실이 가장 작은 T를 후보 가운데 고른다."""
    losses = [F.cross_entropy(z / T, y).item() for T in TEMPS]
    return TEMPS[int(np.argmin(losses))].item()

def members(epoch):
    """시드 10개의 epoch번째 체크포인트를 읽어 모델 10개를 만든다."""
    made = []
    for seed in range(SEEDS):
        net = make_cnn()
        net.load_state_dict(torch.load(f"ckpt/seed{seed}_e{epoch:02d}.pt"))
        made.append(net)
    return made

def summary(nets, x, y, T=None):
    """모델 하나씩의 정확도와 앙상블의 정확도, 손실, ECE. T가 없으면 여기서 고른다."""
    probs = torch.stack([torch.softmax(scores_of(net, x), dim=1) for net in nets])
    hits = (probs.argmax(dim=2) == y).numpy()                # (모델, 이미지)
    together = probs.mean(dim=0)                             # 앙상블의 확률
    T = fit_T(torch.log(together), y) if T is None else T
    cooled = torch.softmax(torch.log(together) / T, dim=1)
    joint = (cooled.argmax(dim=1) == y).numpy()[None]        # 앙상블이 맞혔는가
    return dict(hits=hits, single=hits.mean(axis=1) * 100, cooled=cooled, T=T,
                joint=joint, together=float(joint.mean() * 100),
                nll=nll(cooled, y), ece=ece(cooled, y), raw_ece=ece(together, y))

by_epoch = acc.mean(axis=0)                                  # 에폭마다 시드 10개의 평균
EPOCH = int(by_epoch.argmax()) + 1                           # 검증 자료로 고른 에폭
finals = members(EPOCH)
on_valid = summary(finals, X_va, y_va)                       # 온도도 검증 자료로 고른다
T_FILES = ["t10k-images-idx3-ubyte.gz", "t10k-labels-idx1-ubyte.gz"]
plan = {"모델": f"합성곱 16-32, 시드 0~9, {EPOCH}에폭의 체크포인트",
        "고른 것": {"에폭": EPOCH, "온도": round(on_valid["T"], 3)},
        "보고할 값": ["모델 하나의 정확도(시드 10개)", "앙상블의 정확도",
                      "앙상블의 손실과 ECE(온도 조정 뒤)", "95% 구간"],
        "시험용 파일": {name: fingerprint(name) for name in T_FILES},
        "평가 횟수": 1}
with open("plan.json", "w", encoding="utf-8") as f:
    json.dump(plan, f, ensure_ascii=False, indent=1)
print(open("plan.json", encoding="utf-8").read())
print(" ".join(f"{a:.2f}" for a in by_epoch[9:]))

In [ ]:
def interval(hits, draws=2000):
    """hits(모델, 자료)에서 자료와 모델을 함께 복원 추출한 정확도의 95% 구간."""
    rng, made = np.random.default_rng(0), []
    n_models, n = hits.shape
    for _ in range(draws):
        rows, models = rng.integers(0, n, n), rng.integers(0, n_models, n_models)
        made.append(hits[models][:, rows].mean() * 100)
    return np.percentile(made, [2.5, 97.5])

# 시험용 파일을 여기서 처음 읽는다. 고를 것은 위의 셀에서 모두 정했다.
X_te, y_te, _, _ = tensors(read_idx(T_FILES[0], 16).reshape(-1, 28, 28),
                           read_idx(T_FILES[1], 8).astype(np.int64),
                           pixel_mean, pixel_std)
on_test = summary(finals, X_te, y_te, T=on_valid["T"])       # 온도는 고르지 않는다
print(tuple(X_te.shape), np.unique(np.bincount(y_te.numpy())).tolist())

def describe(w):
    """summary의 결과를 표의 한 열로 만든다. 구간도 계산해 w에 넣는다."""
    w["span"], w["span_joint"] = interval(w["hits"]), interval(w["joint"])
    one = w["single"]
    return [f"{np.median(one):.2f}", f"{one.min():.2f}~{one.max():.2f}",
            f"{w['span'][0]:.2f}~{w['span'][1]:.2f}", f"{w['together']:.2f}",
            f"{w['span_joint'][0]:.2f}~{w['span_joint'][1]:.2f}", f"{w['nll']:.4f}",
            f"{w['ece']:.2f}", f"{w['raw_ece']:.2f}"]

NAMES = ["모델 하나, 중앙값", "모델 하나, 범위", "모델 하나, 95% 구간",
         "앙상블의 정확도", "앙상블, 95% 구간", "앙상블의 손실",
         "앙상블의 ECE, 온도 조정 뒤", "앙상블의 ECE, 온도 조정 전"]
print(pad("", 28) + "검증 10,000장   시험 10,000장")
for name, here, there in zip(NAMES, describe(on_valid), describe(on_test)):
    print(pad(name, 28) + f"{here:>13} {there:>15}")
print(round(float(np.median(on_test["single"]) - np.median(on_valid["single"])), 2),
      round(on_test["together"] - on_valid["together"], 2))
p = on_test["together"] / 100                                # 식으로 구한 구간과 비교
se = math.sqrt(p * (1 - p) / len(y_te)) * 100
print(round(se, 2), round(p * 100 - 1.96 * se, 2), round(p * 100 + 1.96 * se, 2))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

reviews = pd.read_csv("ratings_train.txt", sep="\t").dropna(subset=["document"])
reviews = reviews.reset_index(drop=True)
shuffled = np.random.default_rng(0).permutation(len(reviews))    # 11장과 같은 나누기
part_tr, part_va = reviews.iloc[shuffled[:30000]], reviews.iloc[shuffled[-10000:]]
texts_tr, texts_va = part_tr["document"].tolist(), part_va["document"].tolist()
texts_un = reviews.iloc[shuffled[30000:-10000]]["document"].tolist()  # 레이블 없이 씀
lab_tr = torch.tensor(part_tr["label"].to_numpy())
lab_va = torch.tensor(part_va["label"].to_numpy())

PAD, UNKNOWN = 0, 1                                 # 0번은 패딩, 1번은 어휘에 없는 글자
counts = Counter(ch for text in texts_tr for ch in text)
vocab = {ch: k + 2 for k, (ch, n) in enumerate(counts.most_common()) if n >= 5}
V = len(vocab) + 2
START, END = V, V + 1                               # 문장의 시작과 끝을 알리는 토큰

def encode(texts, length=64):
    """<시작> + 글자 번호 + <끝>. length + 2칸이고 남는 칸은 0이다."""
    ids = torch.zeros(len(texts), length + 2, dtype=torch.long)
    for row, text in enumerate(texts):
        found = [START] + [vocab.get(ch, UNKNOWN) for ch in text][:length] + [END]
        ids[row, :len(found)] = torch.tensor(found, dtype=torch.long)
    return ids

ids_tr, ids_va, ids_un = encode(texts_tr), encode(texts_va), encode(texts_un)
grams = TfidfVectorizer(analyzer="char", ngram_range=(1, 3), min_df=2,
                        sublinear_tf=True)
logit = LogisticRegression(max_iter=1000, C=3)              # 11장의 기준선
logit.fit(grams.fit_transform(texts_tr), lab_tr)
hits_va = {"로지스틱 회귀": (logit.predict(grams.transform(texts_va))
                             == lab_va.numpy())[None]}
print(len(texts_tr), len(texts_va), len(texts_un), V + 2)
print(round(float(hits_va["로지스틱 회귀"].mean() * 100), 2))

In [ ]:
class SelfAttention(nn.Module):
    """12장의 셀프 어텐션. x: (배치, 길이, dim)."""
    def __init__(self, dim, heads):
        super().__init__()
        self.heads = heads
        self.to_q, self.to_k, self.to_v = (nn.Linear(dim, dim) for _ in range(3))
        self.out = nn.Linear(dim, dim)

    def forward(self, x, hide, causal):
        B, L, D = x.shape
        H = self.heads
        split = lambda t: t.view(B, L, H, D // H).transpose(1, 2)    # (B, H, L, D/H)
        q, k, v = split(self.to_q(x)), split(self.to_k(x)), split(self.to_v(x))
        scores = q @ k.transpose(-2, -1) / math.sqrt(D // H)         # (B, H, L, L)
        scores = scores.masked_fill(hide[:, None, None, :], float("-inf"))
        if causal:                                                  # 뒤의 토큰을 가린다
            later = torch.triu(torch.ones(L, L, dtype=torch.bool), diagonal=1)
            scores = scores.masked_fill(later, float("-inf"))
        mixed = torch.softmax(scores, dim=-1) @ v
        return self.out(mixed.transpose(1, 2).reshape(B, L, D))

class Block(nn.Module):
    """층 정규화 → 셀프 어텐션 → 더하기, 층 정규화 → 완전 연결 층 둘 → 더하기."""
    def __init__(self, dim, heads):
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.attend = SelfAttention(dim, heads)
        self.mlp = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(),
                                 nn.Linear(4 * dim, dim))

    def forward(self, x, hide, causal):
        x = x + self.attend(self.norm1(x), hide, causal)
        return x + self.mlp(self.norm2(x))

class Body(nn.Module):
    """몸통: 토큰 임베딩 + 위치 임베딩 → 블록 2개 → 층 정규화. 출력 층은 없다."""
    def __init__(self, dim=64, heads=4, layers=2, length=66):
        super().__init__()
        self.table = nn.Embedding(V + 2, dim)
        self.where = nn.Embedding(length, dim)
        self.blocks = nn.ModuleList(Block(dim, heads) for _ in range(layers))
        self.norm = nn.LayerNorm(dim)

    def forward(self, ids, causal):                          # (배치, 길이, 64)
        x = self.table(ids) + self.where(torch.arange(ids.shape[1]))
        hide = ids == PAD
        for block in self.blocks:
            x = block(x, hide, causal)
        return self.norm(x)

class LanguageModel(nn.Module):
    """몸통 + 다음 토큰의 점수 V + 2개를 내는 출력 층."""
    def __init__(self):
        super().__init__()
        self.body, self.head = Body(), nn.Linear(64, V + 2)

    def forward(self, ids):                                  # (배치, 길이, V + 2)
        return self.head(self.body(ids, causal=True))

class Classifier(nn.Module):
    """몸통 + 긍정과 부정의 점수 2개를 내는 출력 층."""
    def __init__(self, causal=True):
        super().__init__()
        self.body, self.head, self.causal = Body(), nn.Linear(64, 2), causal

    def forward(self, ids):
        x = self.body(ids, self.causal)
        real = (ids != PAD)[:, :, None].float()              # 패딩을 빼고 평균한다
        return self.head((x * real).sum(dim=1) / real.sum(dim=1))

def batches(lengths, seed, size=64, chunk=50):
    """길이가 비슷한 문장끼리 배치를 만든다. 섞은 뒤 3,200문장씩 길이 순으로 놓는다."""
    shuffler = torch.Generator().manual_seed(seed)
    mixed = torch.randperm(len(lengths), generator=shuffler)
    made = []
    for start in range(0, len(mixed), size * chunk):
        part = mixed[start:start + size * chunk]
        part = part[torch.argsort(lengths[part])]
        made += [part[k:k + size] for k in range(0, len(part), size)]
    return [made[k] for k in torch.randperm(len(made), generator=shuffler)]

def fit(net, X, y, lr, seed=0, epochs=2, loss_of=None):
    """AdamW, 배치 크기 64. 학습률은 처음 10% 걸음 동안 올리고 코사인으로 줄인다."""
    optimizer = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=0.0)
    lengths = (X != PAD).sum(dim=1)
    total, step = epochs * math.ceil(len(X) / 64), 0
    warm = max(1, total // 10)
    for epoch in range(epochs):
        net.train()
        for rows in batches(lengths, seed * 1000 + epoch):
            step += 1
            wave = 0.5 * (1 + math.cos(math.pi * step / total))
            for group in optimizer.param_groups:
                group["lr"] = lr * min(1, step / warm) * wave
            ids = X[rows][:, :int(lengths[rows].max())]    # 배치에서 가장 긴 문장까지만
            optimizer.zero_grad()
            if loss_of is None:
                loss = F.cross_entropy(net(ids), y[rows])
            else:
                loss = loss_of(net, ids)
            loss.backward()
            optimizer.step()
    return net

def lm_loss(net, ids):
    """앞 토큰들로 다음 토큰을 맞히는 교차 엔트로피. 패딩인 목표는 뺀다."""
    out = net(ids[:, :-1])
    return F.cross_entropy(out.reshape(-1, out.shape[-1]), ids[:, 1:].reshape(-1),
                           ignore_index=PAD)

def text_hits(net, X, y):
    """문장마다 맞혔는지."""
    net.eval()
    with torch.no_grad():
        guess = torch.cat([net(X[k:k + 1000]).argmax(dim=1)
                           for k in range(0, len(X), 1000)])
    return (guess == y).numpy()

lap = time.perf_counter()
torch.manual_seed(0)
lm = fit(LanguageModel(), ids_un, None, 0.03, epochs=2, loss_of=lm_loss)   # 13.2절
torch.save({"body": lm.body.state_dict(), "sentences": len(ids_un), "lr": 0.03,
            "epochs": 2, "seed": 0}, "body_pretrained.pt")
took["언어 모델 사전 학습"] = time.perf_counter() - lap
lap = time.perf_counter()
tuned = []
for seed in range(5):                                        # 13.4절의 미세 조정
    torch.manual_seed(seed)
    net = Classifier()
    net.body.load_state_dict(torch.load("body_pretrained.pt")["body"])
    tuned.append(fit(net, ids_tr, lab_tr, 0.01, seed, epochs=2))
took["미세 조정 5번"] = time.perf_counter() - lap
hits_va["미세 조정"] = np.array([text_hits(net, ids_va, lab_va) for net in tuned])
print(" ".join(f"{h.mean() * 100:.2f}" for h in hits_va["미세 조정"]))
print(round(float(np.median(hits_va["미세 조정"].mean(axis=1)) * 100), 2),
      round(float(hits_va["로지스틱 회귀"].mean() * 100), 2))

In [ ]:
def gap_interval(one, other, draws=2000):
    """정확도의 차이(one - other, %p)와 95% 구간. 문장과 모델을 함께 복원 추출한다."""
    rng, made = np.random.default_rng(0), []
    n = one.shape[1]
    for _ in range(draws):
        rows = rng.integers(0, n, n)
        mine = one[rng.integers(0, len(one), len(one))][:, rows].mean()
        made.append((mine - other[:, rows].mean()) * 100)
    return (one.mean() - other.mean()) * 100, np.percentile(made, [2.5, 97.5])

# 영화평의 시험용 파일을 여기서 처음 읽는다.
exam = pd.read_csv("ratings_test.txt", sep="\t").dropna(subset=["document"])
texts_te, lab_te = exam["document"].tolist(), torch.tensor(exam["label"].to_numpy())
ids_te = encode(texts_te)
hits_te = {"로지스틱 회귀": (logit.predict(grams.transform(texts_te))
                             == lab_te.numpy())[None],
           "미세 조정": np.array([text_hits(net, ids_te, lab_te) for net in tuned])}
seen = set(texts_tr)
print(len(texts_te), round(float(lab_te.float().mean()), 4),
      sum(text in seen for text in texts_te))
print(pad("", 22) + "검증 10,000문장      시험 " + f"{len(texts_te):,}문장")
spans = {}
for name in hits_te:
    cells = []
    for hits in (hits_va[name], hits_te[name]):
        low, high = interval(hits)
        cells.append(f"{hits.mean() * 100:.2f} ({low:.2f}~{high:.2f})")
        spans[name, hits.shape[1]] = (hits.mean() * 100, low, high)
    print(pad(name, 22) + f"{cells[0]:>20} {cells[1]:>22}")
print(" ".join(f"{h.mean() * 100:.2f}" for h in hits_te["미세 조정"]))
for label, group in [("검증", hits_va), ("시험", hits_te)]:
    gap, (low, high) = gap_interval(group["미세 조정"], group["로지스틱 회귀"])
    spans["차이", label] = (gap, low, high)
    print(f"{label}: 미세 조정 - 로지스틱 회귀 {gap:+.2f} [{low:+.2f}, {high:+.2f}]")

## 16.5 시간이 지나면 달라지는 자료와 다시 학습하는 기준

In [ ]:
torch.set_num_threads(1)                                     # 8장과 같은 설정

def period(path):
    """파일 머리말에서 계약일자 범위를 읽는다. 실거래가 파일이 아니면 빈 리스트."""
    with open(path, encoding="cp949", errors="ignore") as f:
        head = "".join(f.readline() for _ in range(15))
    return re.findall(r"계약일자 : (\d{4}-\d{2}-\d{2}) ~ (\d{4}-\d{2}-\d{2})", head)

files = sorted(f for f in os.listdir(".") if f.endswith(".csv") and period(f))
raw = pd.concat([pd.read_csv(f, encoding="cp949", skiprows=15, thousands=",",
                             dtype={"번지": str, "본번": str, "부번": str, "동": str})
                 for f in files], ignore_index=True)
v = raw[(raw["해제사유발생일"] == "-") & (raw["거래유형"] != "직거래")].copy()
v = v.rename(columns={"전용면적(㎡)": "면적"})
v["억"] = v["거래금액(만원)"] / 1e4
v["날짜"] = pd.to_datetime(v["계약년월"].astype(str)
                           + v["계약일"].astype(str).str.zfill(2), format="%Y%m%d")
v["동이름"] = v["시군구"].str.split().str[1] + " " + v["시군구"].str.split().str[2]
v["단지"] = v["시군구"] + " " + v["번지"] + " " + v["단지명"]
v["달"] = (v["날짜"].dt.year - 2023) * 12 + v["날짜"].dt.month    # 2023년 1월이 1
v = v.sort_values("날짜").reset_index(drop=True)

def price(df):
    return np.log(df["억"].to_numpy())

def off(actual_log, predicted_log):
    """오차율의 중앙값(%)."""
    return float(np.median(np.abs(np.exp(predicted_log - actual_log) - 1)) * 100)

def encoder(made):
    """made의 동과 단지에 1부터 번호를 매긴다. 0번은 made에 없던 동과 단지가 쓴다.
    made의 마지막 달보다 뒤의 달은 마지막 달로 바꿔 넣는다(8.4절)."""
    dongs = pd.Index(sorted(made["동이름"].unique()))
    complexes = pd.Index(sorted(made["단지"].unique()))
    last = made["달"].max()
    def numbers(df):
        when = np.minimum(df["달"], last)
        cols = [np.log(df["면적"]), df["층"], df["건축년도"], when]
        return np.column_stack(cols).astype(np.float64)
    center, scale = numbers(made).mean(axis=0), numbers(made).std(axis=0)
    def encode_deals(df):
        x = torch.tensor((numbers(df) - center) / scale, dtype=torch.float32)
        dong = torch.tensor(dongs.get_indexer(df["동이름"]) + 1)   # 없으면 0
        complex_ = torch.tensor(complexes.get_indexer(df["단지"]) + 1)
        return x, dong, complex_
    return encode_deals, len(dongs) + 1, len(complexes) + 1

class TabNet(nn.Module):
    """숫자 열과 동 임베딩 8개, 단지 임베딩 16개를 이어 붙여 은닉층 2개에 넣는다."""
    def __init__(self, n_dong, n_complex, n_numbers, hidden=64):
        super().__init__()
        self.dong = nn.Embedding(n_dong, 8)
        self.complex = nn.Embedding(n_complex, 16)
        self.body = nn.Sequential(nn.Linear(n_numbers + 8 + 16, hidden), nn.ReLU(),
                                  nn.Linear(hidden, hidden), nn.ReLU(),
                                  nn.Linear(hidden, 1))

    def forward(self, x, dong, complex_):
        joined = torch.cat([x, self.dong(dong), self.complex(complex_)], dim=1)
        return self.body(joined)[:, 0]

def train(made, drop=0.1, epochs=30, seed=0):
    """8.3절의 학습. 거래 표를 받아 로그 가격을 예측하는 함수를 돌려준다."""
    encode_deals, n_dong, n_complex = encoder(made)
    x, dong, complex_ = encode_deals(made)
    target = price(made)
    mean, std = target.mean(), target.std()
    z = torch.tensor((target - mean) / std, dtype=torch.float32)
    torch.manual_seed(seed)
    net = TabNet(n_dong, n_complex, x.shape[1])
    optimizer = torch.optim.Adam(net.parameters(), lr=0.003)
    steps = epochs * -(-len(z) // 256)                       # 배치 크기 256
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, steps)
    shuffler = torch.Generator().manual_seed(seed)
    for epoch in range(epochs):
        net.train()
        mixed = torch.randperm(len(z), generator=shuffler)
        for begin in range(0, len(z), 256):
            rows = mixed[begin:begin + 256]
            hide = torch.rand(len(rows), generator=shuffler) < drop
            picked = complex_[rows].masked_fill(hide, 0)     # 처음 보는 단지로 바꾼다
            optimizer.zero_grad()
            F.mse_loss(net(x[rows], dong[rows], picked), z[rows]).backward()
            optimizer.step()
            scheduler.step()
    net.eval()
    def predict(df):
        with torch.no_grad():
            return net(*encode_deals(df)).numpy().astype(np.float64) * std + mean
    return predict

def month_error(model, month):
    deals = v[v["달"] == month]
    return off(price(deals), model(deals))

a23 = v[v["달"] <= 12]
early, late = a23[a23["달"] <= 9], a23[a23["달"] >= 10]     # 8.5절의 검증
valid_error = off(price(late), train(early)(late))
middle = v.groupby("달")["억"].median()                      # 달마다 거래 가격의 중앙값
print(len(a23), len(v) - len(a23), round(valid_error, 2))
for year in (0, 1):
    print(" ".join(f"{m:.2f}" for m in middle.iloc[year * 12:(year + 1) * 12]))

In [ ]:
LIMIT = valid_error * 1.25                 # 다시 학습하는 기준: 검증 오차율의 1.25배
lap = time.perf_counter()
models = {13: train(a23)}                  # 2023년으로 학습시킨 모델
for month in range(14, 25):                # 그 달이 오기 전까지의 거래로 학습시킨 모델
    models[month] = train(v[v["달"] < month])
took["아파트 모델 12번"] = time.perf_counter() - lap

def follow(rule):
    """2024년 열두 달을 차례로 지나며 (달, 그 달의 오차율, 다시 학습했는가)를 적는다.
    그 달의 오차율은 달이 끝나 실제 가격이 모인 뒤에 알 수 있다."""
    model, seen_error, rows = models[13], valid_error, []
    for month in range(13, 25):
        again = month > 13 and (rule == "매달" or
                                (rule == "기준" and seen_error > LIMIT))
        if again:
            model = models[month]          # 지난달까지의 거래로 다시 학습시킨 모델
        seen_error = month_error(model, month)
        rows.append((month, seen_error, again))
    return rows

paths = {rule: follow(rule) for rule in ["고정", "기준", "매달"]}
print(f"기준 {LIMIT:.2f}%")
print(pad("", 7) + "".join(f"{m - 12:>4}월" for m in range(13, 25)))
for rule, rows in paths.items():
    print(pad(rule, 7) + "".join(f"{e:>6.2f}" for _, e, _ in rows))
for rule, rows in paths.items():
    errors = [e for _, e, _ in rows]
    print(pad(rule, 7) + f"평균 {np.mean(errors):.2f}  가장 큰 달 {max(errors):.2f}  "
          f"다시 학습 {sum(a for _, _, a in rows)}번")
print("기준 방침이 다시 학습시킨 달:", [m - 12 for m, _, a in paths["기준"] if a])
torch.set_num_threads(2)

In [ ]:
import sklearn

print(sys.version.split()[0], np.__version__, pd.__version__, torch.__version__,
      sklearn.__version__)
print(platform.machine(), torch.get_default_dtype(), torch.get_num_threads())
for name, seconds in took.items():                           # 걸린 시간
    print(pad(name, 24) + f"{seconds:>5.0f}초")

In [ ]:
!pip -q install koreanize-matplotlib

In [ ]:
import matplotlib.pyplot as plt
import koreanize_matplotlib
from matplotlib.colors import to_rgba
from matplotlib.ticker import FuncFormatter

NAVY, BRICK, TEAL, GRAY = "#16324f", "#b5532a", "#2a7f73", "#9aa1ab"
BOX = {"fc": "white", "ec": "none", "alpha": 0.9, "pad": 1.5}


def frame(ax, title=None, color=NAVY, grid="y"):
    """그래프마다 반복하는 꾸밈: 제목, 보조선, 테두리 정리."""
    if title:
        ax.set_title(title, fontsize=9, loc="left", color=color, weight="bold")
    ax.grid(axis=grid, color="#e3e7ec", lw=0.6)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=8)
    ax.spines[["top", "right"]].set_visible(False)


def plain_ticks(axis):
    """로그 축의 눈금을 10의 거듭제곱 대신 보통 숫자로 적는다."""
    axis.set_major_formatter(FuncFormatter(lambda val, _: f"{val:,.0f}" if val >= 1 else f"{val:.7f}".rstrip("0")))

In [ ]:
# 그림 16-3
fig, axes = plt.subplots(2, 1, figsize=(7, 9.2), gridspec_kw={"height_ratios": [1.5, 1]})
ax = axes[0]
for name, shade, mark4 in [("스레드 1개", NAVY, "o"), ("64비트 실수", BRICK, "s")]:
    ax.plot(STOPS, apart[name], color=shade, marker=mark4, lw=1.8, ms=6.5, label=name + "로 바꿔 실행한 모델과의 차이")
for k in [0, 3, 6, 9]:
    ax.text(STOPS[k] * 1.12, apart["스레드 1개"][k] / 2.6, f"{apart['스레드 1개'][k]:.1e}", fontsize=9.6, color=NAVY, weight="bold")
    ax.text(STOPS[k] / 1.1, apart["64비트 실수"][k] * 2.1, f"{apart['64비트 실수'][k]:.1e}", fontsize=9.6, color=BRICK, weight="bold", ha="right")
ax.axvline(94, color="#888", lw=0.8, ls="--"); ax.text(94, 2.2e-9, "1에폭", ha="center", fontsize=9, color="#444", bbox=BOX)
ax.axvline(940, color="#888", lw=0.8, ls="--"); ax.text(940, 2.2e-9, "10에폭", ha="center", fontsize=9, color="#444", bbox=BOX)
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlim(0.35, 3200); ax.set_ylim(1e-9, 2)
ax.set_xticks([1, 10, 100, 1000]); plain_ticks(ax.xaxis)
ax.yaxis.set_major_formatter(FuncFormatter(lambda val, _: f"{val:.0e}")); ax.yaxis.set_minor_formatter(FuncFormatter(lambda val, _: ""))
ax.set_xlabel("가중치를 고친 걸음의 수 (로그 축)", fontsize=9); ax.set_ylabel("가중치의 가장 큰 차이 (로그 축)", fontsize=9)
ax.legend(frameon=False, fontsize=9.4, loc="upper left")
frame(ax)
ax.set_title("① 같은 시드로 시작한 두 실행의 가중치가 벌어지는 속도", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.tick_params(labelsize=9)
ax = axes[1]
rows4 = [("같은 기록으로 다시", GRAY), ("스레드 1개", NAVY), ("64비트 실수", BRICK)]
for spot, (name, shade) in enumerate(rows4):
    gap4, mark4, moved4 = changed[name]
    ax.barh(spot, moved4, color=shade, height=0.56)
    ax.text(moved4 + 1.2, spot, f"{moved4}장   정확도 {mark4:.2f}%   가중치 차이 {gap4:.1e}", va="center", fontsize=10, color="#111", weight="bold")
ax.set_yticks(range(3)); ax.set_yticklabels([name for name, _ in rows4], fontsize=9.6); ax.invert_yaxis()
ax.set_xlim(0, 130); ax.set_xlabel("10에폭 뒤 검증 10,000장 가운데 예측이 달라진 이미지의 수", fontsize=9)
frame(ax, grid="x")
ax.set_title("② 10에폭이 끝났을 때의 차이", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.tick_params(axis="x", labelsize=9); ax.tick_params(axis="y", length=0)
fig.subplots_adjust(left=0.2, right=0.97, top=0.965, bottom=0.06, hspace=0.24)
pos = axes[0].get_position(); axes[0].set_position([0.11, pos.y0, 0.86, pos.height])
fig.savefig("fig_16_3.png", dpi=300)
plt.show()
print([f"{v:.1e}" for v in apart["스레드 1개"]])

In [ ]:
# 그림 16-4
fig, axes = plt.subplots(1, 2, figsize=(7, 8.6), gridspec_kw={"width_ratios": [1.25, 1]})
groups3 = [("체크포인트에서 이어서 학습", [(name, resumed[name]) for name in WAYS], TEAL),
           ("같은 기록, 다른 환경", [(name, changed[name]) for name in VARIANTS], NAVY),
           ("기록에서 한 항목이 다름", [(name, changed[name]) for name in MISSING], BRICK)]
height, ticks3 = 0, []
for title, rows3, shade in groups3:
    axes[0].text(-600, height - 0.72, title, fontsize=10, color=shade, weight="bold", va="center")
    for name, (gap3, mark3, moved3) in rows3:
        axes[0].barh(height, moved3, color=shade, height=0.62)
        axes[0].text(moved3 + 14, height, f"{moved3:,}장" if moved3 else "0장 (완전히 같다)", va="center", fontsize=10, color="#111", weight="bold")
        axes[0].text(-12, height, name, va="center", ha="right", fontsize=9.2, color="#222")
        axes[1].scatter([mark3], [height], s=58, color="white" if mark3 != changed["같은 기록으로 다시"][1] else shade, edgecolor=shade, lw=1.8, zorder=5)
        axes[1].text(mark3 + (0.1 if mark3 >= 86.89 else -0.1), height, f"{mark3:.2f}", va="center", ha="left" if mark3 >= 86.89 else "right", fontsize=10, color="#111", weight="bold")
        height += 1
    height += 1.25
for ax in axes:
    ax.set_ylim(height - 1.6, -1.5); ax.set_yticks([])
axes[0].set_xlim(-600, 1050); axes[0].set_xticks([0, 250, 500, 750, 1000])
axes[0].set_xlabel("검증 10,000장 가운데 예측이 달라진 이미지의 수", fontsize=9)
axes[1].axvline(changed["같은 기록으로 다시"][1], color="#666", lw=0.9, ls="--")
axes[1].text(changed["같은 기록으로 다시"][1], -1.15, f"기록의 숫자 {changed['같은 기록으로 다시'][1]:.2f}%", ha="center", fontsize=9.2, color="#333")
axes[1].set_xlim(85.2, 88.6); axes[1].set_xlabel("검증 정확도 (%)", fontsize=9)
frame(axes[0], grid="x"); frame(axes[1], grid="x")
axes[0].spines["left"].set_visible(False)
fig.text(0.04, 0.972, "① 기준 모델과 예측이 달라진 이미지", fontsize=10, color=NAVY, weight="bold")
axes[1].set_title("② 검증 정확도", fontsize=10, loc="left", color=NAVY, weight="bold")
for ax in axes:
    ax.tick_params(axis="x", labelsize=9)
fig.subplots_adjust(left=0.04, right=0.97, top=0.955, bottom=0.07, wspace=0.06)
fig.savefig("fig_16_4.png", dpi=300)
plt.show()
print([moved3 for _, rows3, _ in groups3 for _, (_, _, moved3) in rows3])

In [ ]:
# 그림 16-5
fig, axes = plt.subplots(2, 1, figsize=(7, 9.2), gridspec_kw={"height_ratios": [1.35, 1]})
ax = axes[0]
epochs5 = np.arange(1, EPOCHS + 1)
top5, low5 = int(acc[:, -1].argmax()), int(acc[:, -1].argmin())
for seed in range(SEEDS):
    if seed not in (top5, low5):
        ax.plot(epochs5, acc[seed], color=GRAY, lw=1.0, alpha=0.8)
ax.plot(epochs5, acc[top5], color=BRICK, lw=1.8, label=f"20에폭에서 가장 높은 시드 {top5}")
ax.plot(epochs5, acc[low5], color=TEAL, lw=1.8, label=f"20에폭에서 가장 낮은 시드 {low5}")
ax.plot(epochs5, acc.mean(axis=0), color=NAVY, lw=2.6, label="시드 10개의 평균")
for k in [9, 14, 19]:
    ax.text(epochs5[k], acc.mean(axis=0)[k] - 0.62, f"{acc.mean(axis=0)[k]:.2f}", ha="center", fontsize=9.8, color=NAVY, weight="bold", bbox=BOX)
ax.text(20.25, acc[top5, -1], f"{acc[top5, -1]:.2f}", va="center", fontsize=9.8, color=BRICK, weight="bold")
ax.text(20.25, acc[low5, -1], f"{acc[low5, -1]:.2f}", va="center", fontsize=9.8, color=TEAL, weight="bold")
ax.set_xlim(1, 21.8); ax.set_ylim(83.5, 89); ax.set_xticks([1, 5, 10, 15, 20])
ax.set_xlabel("에폭", fontsize=9); ax.set_ylabel("검증 10,000장의 정확도 (%)", fontsize=9)
ax.legend(frameon=False, fontsize=9.4, loc="lower right")
frame(ax)
ax.set_title("① 시드 10개의 검증 정확도 (1~4에폭은 83.5% 아래여서 일부가 보이지 않는다)", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.tick_params(labelsize=9)
ax = axes[1]
sorted5 = np.argsort(-acc[:, -1])
for spot, seed in enumerate(sorted5):
    shade = BRICK if seed == top5 else TEAL if seed == low5 else NAVY
    ax.bar(spot, acc[seed, -1] - 85, bottom=85, color=shade, width=0.66)
    ax.text(spot, acc[seed, -1] + 0.07, f"{acc[seed, -1]:.2f}", ha="center", fontsize=9.6, color="#111", weight="bold")
middle5 = float(np.median(acc[:, -1]))
ax.axhline(middle5, color="#333", lw=0.9, ls="--"); ax.text(9.62, middle5 + 0.07, f"중앙값 {middle5:.2f}", ha="right", fontsize=9.4, color="#333", bbox=BOX)
ax.set_xticks(range(SEEDS)); ax.set_xticklabels([f"시드 {seed}" for seed in sorted5], fontsize=9); ax.set_ylim(85, 88.9); ax.set_xlim(-0.6, 9.7)
ax.set_ylabel("20에폭의 검증 정확도 (%)", fontsize=9)
frame(ax)
ax.set_title(f"② 20에폭의 정확도를 높은 순서로: 가장 높은 값과 중앙값의 차이 {acc[:, -1].max() - middle5:.2f}%p", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.tick_params(axis="y", labelsize=9); ax.tick_params(axis="x", length=0)
fig.subplots_adjust(left=0.1, right=0.97, top=0.965, bottom=0.05, hspace=0.24)
fig.savefig("fig_16_5.png", dpi=300)
plt.show()
print(top5, low5, round(float(acc[:, -1].max() - middle5), 2))

In [ ]:
# 그림 16-6
fig, axes = plt.subplots(3, 1, figsize=(7, 9.3))
notes6 = {"시드": "20에폭 모델 10개 가운데 가장 높은 하나를 고른다", "체크포인트": "시드마다 20개의 에폭 가운데 가장 높은 에폭을 고른다",
          "시드와 체크포인트": "200개(시드 10 × 에폭 20) 가운데 가장 높은 하나를 고른다"}
for ax, rule, mark6 in zip(axes, RULES, "①②③"):
    for spot, n_pick in enumerate(PICKS):
        here6, there6 = lifted[rule, n_pick]
        ax.bar(spot - 0.2, here6, width=0.36, color=BRICK, label="고르는 데 쓴 이미지에서" if spot == 0 else None)
        ax.bar(spot + 0.2, there6, width=0.36, color=NAVY, label="쓰지 않은 이미지에서" if spot == 0 else None)
        ax.text(spot - 0.2, here6 + 0.04, f"{here6:+.2f}", ha="center", fontsize=10.2, color=BRICK, weight="bold")
        ax.text(spot + 0.2, max(there6, 0) + 0.04, f"{there6:+.2f}", ha="center", fontsize=10.2, color=NAVY, weight="bold")
        ax.text(spot + 0.46, here6 * 0.5, f"차이\n{here6 - there6:.2f}", ha="left", va="center", fontsize=9.2, color="#333", linespacing=1.3)
    ax.axhline(0, color="#333", lw=0.8)
    ax.set_xticks(range(3)); ax.set_xticklabels([f"{n:,}장으로 고름" for n in PICKS], fontsize=10); ax.set_xlim(-0.55, 2.8); ax.set_ylim(-0.15, 2.35)
    ax.set_ylabel("20에폭 평균보다 높은 정도 (%p)", fontsize=9)
    if rule == "시드":
        ax.legend(frameon=False, fontsize=9.4, loc="upper left", ncol=2)
    frame(ax)
    ax.set_title(f"{mark6} {rule}: {notes6[rule]}", fontsize=10, loc="left", color=NAVY, weight="bold")
    ax.tick_params(axis="y", labelsize=9); ax.tick_params(axis="x", length=0)
fig.subplots_adjust(left=0.1, right=0.97, top=0.965, bottom=0.045, hspace=0.3)
fig.savefig("fig_16_6.png", dpi=300)
plt.show()
print([round(float(lifted[rule, n][0] - lifted[rule, n][1]), 2) for rule in RULES for n in PICKS])

In [ ]:
# 그림 16-8
def ladder(ax, rows, shade, title):
    # 확신의 구간마다 맞힌 비율을 막대로 그리고 대각선과 비교한다
    ax.plot([0, 1], [0, 1], color="#666", lw=0.9, ls="--", zorder=1)
    for low, n, sure_, hit in rows:
        if n < 10:
            continue
        ax.bar(low + 0.05, hit, width=0.086, color=shade, zorder=2)
        ax.plot([low + 0.007, low + 0.093], [sure_, sure_], color="#111", lw=1.6, zorder=3)
        ax.text(low + 0.05, hit + 0.02 if hit >= sure_ else hit - 0.02, f"{hit:.2f}", ha="center", va="bottom" if hit >= sure_ else "top", fontsize=9,
                color=shade if hit >= sure_ else "white", weight="bold", zorder=4)
    ax.set_xlim(0.18, 1.01); ax.set_ylim(0, 1.08)
    ax.set_xticks(np.arange(0.2, 1.01, 0.2)); ax.set_xlabel("앙상블이 준 확신", fontsize=9)
    frame(ax)
    ax.set_title(title, fontsize=10, loc="left", color=NAVY, weight="bold")
    ax.tick_params(labelsize=9)

fig = plt.figure(figsize=(7, 9.4))
grid = fig.add_gridspec(2, 2, height_ratios=[1.25, 1], hspace=0.3, wspace=0.2, left=0.1, right=0.97, top=0.96, bottom=0.06)
ax = fig.add_subplot(grid[0, :])
for seed in range(SEEDS):
    ax.plot([0, 1], [on_valid["single"][seed], on_test["single"][seed]], color=GRAY, lw=1.0, marker="o", ms=4.5, zorder=2)
for spot, (where8, label8) in enumerate([(on_valid, "검증 10,000장"), (on_test, "시험 10,000장")]):
    side = -1 if spot == 0 else 1
    ax.plot([spot + side * 0.13] * 2, where8["span"], color=NAVY, lw=2.6, zorder=3)
    ax.scatter([spot + side * 0.13], [np.median(where8["single"])], s=50, color="white", edgecolor=NAVY, lw=1.8, zorder=5)
    ax.text(spot + side * 0.17, np.median(where8["single"]), f"모델 하나\n중앙값 {np.median(where8['single']):.2f}\n구간 {where8['span'][0]:.2f}~{where8['span'][1]:.2f}", ha="right" if spot == 0 else "left",
            va="center", fontsize=9.4, color=NAVY, weight="bold", linespacing=1.45)
    ax.plot([spot + side * 0.13] * 2, where8["span_joint"], color=BRICK, lw=2.6, zorder=3)
    ax.scatter([spot + side * 0.13], [where8["together"]], s=50, color="white", edgecolor=BRICK, lw=1.8, zorder=5)
    ax.text(spot + side * 0.17, where8["together"] + 0.25, f"앙상블 {where8['together']:.2f}\n구간 {where8['span_joint'][0]:.2f}~{where8['span_joint'][1]:.2f}", ha="right" if spot == 0 else "left", va="center",
            fontsize=9.4, color=BRICK, weight="bold", linespacing=1.45)
ax.plot([0, 1], [on_valid["together"], on_test["together"]], color=BRICK, lw=1.2, ls=":", zorder=2)
ax.text(-0.72, 84.95, "회색 선: 시드 하나하나의 모델", fontsize=9, color="#555")
ax.set_xticks([0, 1]); ax.set_xticklabels(["검증 10,000장\n(고르는 데 썼다)", "시험 10,000장\n(한 번 읽었다)"], fontsize=10); ax.set_xlim(-0.75, 1.75); ax.set_ylim(84.8, 90.2)
ax.set_ylabel("정확도 (%)", fontsize=9)
frame(ax)
ax.set_title("① 검증 자료와 시험 자료의 정확도: 모델 하나(남색)와 10개의 앙상블(벽돌색)", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.tick_params(axis="y", labelsize=9)
for col, (probs8, shade, title) in enumerate([(on_test["cooled"], TEAL, f"② 시험, 온도 {on_test['T']:.3f}로 조정: ECE {on_test['ece']:.2f}"), (None, GRAY, f"③ 시험, 조정 전: ECE {on_test['raw_ece']:.2f}")]):
    ax = fig.add_subplot(grid[1, col])
    if probs8 is None:
        probs8 = torch.softmax(torch.log(on_test["cooled"]) * on_test["T"], dim=1)       # 온도 조정을 되돌린 확률
    ladder(ax, bins_of(probs8, y_te), shade, title)
    if col == 0:
        ax.set_ylabel("실제로 맞힌 비율", fontsize=9)
fig.savefig("fig_16_8.png", dpi=300)
plt.show()
print(int((on_test["single"] < on_valid["single"]).sum()), round(on_test["raw_ece"], 2))

In [ ]:
# 그림 16-9
fig, axes = plt.subplots(2, 1, figsize=(7, 9.0), gridspec_kw={"height_ratios": [1.5, 1]})
ax = axes[0]
sizes9 = {"검증": hits_va["로지스틱 회귀"].shape[1], "시험": hits_te["로지스틱 회귀"].shape[1]}
for block, (name, shade) in enumerate([("로지스틱 회귀", NAVY), ("미세 조정", BRICK)]):
    for spot, label9 in enumerate(["검증", "시험"]):
        mean9, low9, high9 = spans[name, sizes9[label9]]
        x9 = block * 2.6 + spot
        ax.plot([x9, x9], [low9, high9], color=shade, lw=2.8)
        ax.scatter([x9], [mean9], s=56, color="white", edgecolor=shade, lw=1.8, zorder=5)
        ax.text(x9 + 0.09, mean9, f"{mean9:.2f}\n({low9:.2f}~{high9:.2f})", va="center", fontsize=9.6, color=shade, weight="bold", linespacing=1.4)
        if name == "미세 조정":
            group9 = hits_va if label9 == "검증" else hits_te
            ax.scatter([x9 - 0.12] * 5, group9[name].mean(axis=1) * 100, s=16, color=GRAY, zorder=4)
    ax.text(block * 2.6 + 0.5, 86.75, name + (" (글자 n-그램, 11장)" if block == 0 else " (사전 학습한 트랜스포머, 13장)"), ha="center", fontsize=10, color=shade, weight="bold")
ax.set_xticks([0, 1, 2.6, 3.6]); ax.set_xticklabels([f"검증\n{sizes9['검증']:,}문장", f"시험\n{sizes9['시험']:,}문장"] * 2, fontsize=9.6)
ax.set_xlim(-0.5, 4.75); ax.set_ylim(79.6, 87.2); ax.set_ylabel("정확도 (%)와 95% 구간", fontsize=9)
ax.text(2.42, 79.85, "회색 점: 시드 하나하나", fontsize=9, color="#555")
frame(ax)
ax.set_title("① 영화평 분류의 검증 정확도와 시험 정확도", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.tick_params(axis="y", labelsize=9)
ax = axes[1]
for spot, label9 in enumerate(["검증", "시험"]):
    gap9, low9, high9 = spans["차이", label9]
    ax.plot([low9, high9], [spot, spot], color=TEAL, lw=3.0)
    ax.scatter([gap9], [spot], s=56, color="white", edgecolor=TEAL, lw=1.8, zorder=5)
    ax.text(high9 + 0.1, spot, f"{gap9:+.2f}  [{low9:+.2f}, {high9:+.2f}]".replace("-", "$-$"), va="center", fontsize=10.2, color=TEAL, weight="bold")
ax.axvline(0, color="#333", lw=0.9)
ax.set_yticks([0, 1]); ax.set_yticklabels([f"검증 {sizes9['검증']:,}문장", f"시험 {sizes9['시험']:,}문장"], fontsize=9.8); ax.set_ylim(1.6, -0.6); ax.set_xlim(-5.2, 0.6)
ax.set_xlabel("미세 조정의 정확도 $-$ 로지스틱 회귀의 정확도 (%p). 왼쪽이면 로지스틱 회귀가 낫다", fontsize=9)
frame(ax, grid="x")
ax.set_title("② 두 모델의 차이와 95% 구간 (문장과 시드를 함께 다시 뽑음)", fontsize=10, loc="left", color=NAVY, weight="bold", x=-0.17)
ax.tick_params(axis="x", labelsize=9); ax.tick_params(axis="y", length=0)
fig.subplots_adjust(left=0.1, right=0.97, top=0.965, bottom=0.07, hspace=0.26)
pos = axes[1].get_position(); axes[1].set_position([0.22, pos.y0, 0.75, pos.height])
fig.savefig("fig_16_9.png", dpi=300)
plt.show()
print({key: [round(float(x), 2) for x in value] for key, value in spans.items() if key[0] == "차이"})

In [ ]:
# 그림 16-11
fig, axes = plt.subplots(3, 1, figsize=(7, 9.4), gridspec_kw={"height_ratios": [0.8, 1.3, 0.75]})
ax = axes[0]
months11 = np.arange(1, 25)
prices11 = np.asarray(middle)                              # 달마다 거래 가격의 중앙값 24개
ax.plot(months11, prices11, color=NAVY, lw=1.8, marker="o", ms=4.5)
ax.axvspan(0.5, 12.5, color=NAVY, alpha=0.06, lw=0); ax.axvspan(12.5, 24.5, color=BRICK, alpha=0.06, lw=0)
ax.text(6.5, 10.75, "2023년: 처음 학습에 쓴 거래", ha="center", fontsize=9.4, color=NAVY, weight="bold")
ax.text(18.5, 8.42, "2024년: 모델을 쓰는 기간", ha="center", fontsize=9.4, color=BRICK, weight="bold")
for k in [0, 11, 17, 23]:
    ax.text(months11[k], prices11[k] + 0.2, f"{prices11[k]:.2f}", ha="left" if k == 0 else "center", fontsize=9.6, color=NAVY, weight="bold")
ax.set_xlim(0.5, 24.5); ax.set_ylim(8.2, 11.2); ax.set_xticks(months11); ax.set_xticklabels([f"{(m - 1) % 12 + 1}" for m in months11], fontsize=9)
ax.set_ylabel("거래 가격의 중앙값 (억 원)", fontsize=9)
frame(ax)
ax.set_title("① 달마다 거래 가격의 중앙값 (가로축은 2023년 1월부터의 달)", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.tick_params(axis="y", labelsize=9)
ax = axes[1]
looks11 = {"고정": (GRAY, "s"), "기준": (BRICK, "o"), "매달": (TEAL, "^")}
for rule, rows11 in paths.items():
    shade, mark11 = looks11[rule]
    errors11 = [e for _, e, _ in rows11]
    ax.plot(range(1, 13), errors11, color=shade if rule != "고정" else "#7c8491", lw=1.9, marker=mark11, ms=6, label={"고정": "고정: 다시 학습하지 않는다", "기준": "기준: 지난달 오차율이 기준을 넘으면 다시 학습", "매달": "매달 다시 학습"}[rule])
for k, j in [(0, 0), (7, 5), (11, 11)]:
    ax.text(k + 1, paths["고정"][k][1] + 0.16, f"{paths['고정'][k][1]:.2f}", ha="center", fontsize=9.6, color="#5b6470", weight="bold")
    ax.text(j + 1, paths["매달"][j][1] - 0.3, f"{paths['매달'][j][1]:.2f}", ha="center", fontsize=9.6, color=TEAL, weight="bold")
for month, error11, again11 in paths["기준"]:
    if again11:
        ax.annotate(f"{month - 12}월 초에 다시 학습\n({month - 13}월 오차율 {paths['기준'][month - 14][1]:.2f} > 기준 {LIMIT:.2f})", xy=(month - 12, error11), xytext=(month - 12 - 2.6, 2.5), fontsize=9.2, color=BRICK,
                    weight="bold", linespacing=1.45, arrowprops=dict(arrowstyle="-|>", color=BRICK, lw=1.0))
        ax.text(month - 12 + 0.2, error11 + 0.2, f"{error11:.2f}", fontsize=9.6, color=BRICK, weight="bold")
ax.axhline(LIMIT, color="#333", lw=0.9, ls="--"); ax.text(0.62, LIMIT + 0.1, f"기준 {LIMIT:.2f}% (검증 오차율 {valid_error:.2f}%의 1.25배)", fontsize=9.2, color="#333")
ax.set_xlim(0.5, 12.5); ax.set_ylim(2.2, 8.4); ax.set_xticks(range(1, 13)); ax.set_xticklabels([f"{m}월" for m in range(1, 13)], fontsize=9)
ax.set_ylabel("그 달 거래의 오차율 중앙값 (%)", fontsize=9)
ax.legend(frameon=False, fontsize=9.2, loc="upper left")
frame(ax)
ax.set_title("② 2024년 달마다의 오차율: 세 가지 방침", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.tick_params(axis="y", labelsize=9)
ax = axes[2]
for spot, (rule, rows11) in enumerate(paths.items()):
    errors11 = [e for _, e, _ in rows11]
    shade = looks11[rule][0] if rule != "고정" else "#7c8491"
    ax.barh(spot, np.mean(errors11), color=shade, height=0.58)
    ax.text(np.mean(errors11) + 0.08, spot, f"평균 {np.mean(errors11):.2f}%   가장 큰 달 {max(errors11):.2f}%   다시 학습 {sum(a for _, _, a in rows11)}번", va="center", fontsize=10, color="#111", weight="bold")
ax.set_yticks(range(3)); ax.set_yticklabels(list(paths), fontsize=10); ax.invert_yaxis(); ax.set_xlim(0, 11.5); ax.set_xticks(range(0, 7))
ax.set_xlabel("열두 달 오차율의 평균 (%)", fontsize=9)
frame(ax, grid="x")
ax.set_title("③ 방침마다의 평균 오차율과 다시 학습한 횟수", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.tick_params(axis="x", labelsize=9); ax.tick_params(axis="y", length=0)
fig.subplots_adjust(left=0.1, right=0.97, top=0.965, bottom=0.055, hspace=0.3)
fig.savefig("fig_16_11.png", dpi=300)
plt.show()
print({rule: round(float(np.mean([e for _, e, _ in rows11])), 2) for rule, rows11 in paths.items()})